In [0]:
%sql
CREATE CATALOG IF NOT EXISTS banking;
USE CATALOG banking;
CREATE SCHEMA IF NOT EXISTS landing;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
catalog='banking'
schema_landing='landing'
schema_bronze='bronze'
schema_silver='silver'
schema_gold='gold'
table_customers=f"{catalog}.{schema_landing}.customers"
table_accounts=f"{catalog}.{schema_landing}.accounts"
table_branches=f"{catalog}.{schema_landing}.branches"
table_transactions=f"{catalog}.{schema_landing}.transactions"
path_gateway=f'/Volumes/{catalog}/{schema_landing}/payment_gateway_logs'
volume_credit=f'/Volumes/{catalog}/{schema_landing}/credit_bureau_reports'


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
import uuid
from datetime import datetime

monitoring_table = f"{catalog}.{schema_bronze}.execution_monitoring"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {monitoring_table} (
    run_id STRING,
    notebook_name STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds DOUBLE,
    status STRING,
    error_message STRING,
    rows_processed BIGINT
)
USING DELTA
""")

print(f"Table created : monitoring_table")



In [0]:
#La donnée peut être MODIFIÉE après création ?
  #  OUI → Delta MERGE   (CUSTOMERS, ACCOUNTS)
  #  NON → Watermark     (CREDIT_BUREAU, PAYMENTS)
    #Table petite/stable → Full Load  (BRANCHES)